**TASKS:**

1.   Load College_notes.csv
2.   Embed all notes and index them in ChromaDB.
3.   Accept any student question in input.
4.   Retrieve the top 3 most relevent notes using vector similarity.
5.   Inject retrieved chunks as context into a Groq LLM prompt.
6.   Generate a close embedded answer.





In [1]:
!pip install sentence-transformers chromadb groq pandas -q
print("Installation Completed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently 

In [2]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
from google.colab import userdata

print("All Libraries imported successfully")

All Libraries imported successfully


In [6]:
GROQ_API_KEY = "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"
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq API client Initiated")

Groq API client Initiated


In [7]:
df = pd.read_csv("college_notes.csv")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print("First 3 rows:\n",df.head(3))

Shape: (15, 4)
Columns: ['note_id', 'subject', 'topic', 'content']
First 3 rows:
   note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [8]:
df['content_length']=df['content'].apply(len)
print(df[["topic","content_length"]].to_string(index=False))

                         topic  content_length
                 ETL Pipelines             216
                 SQL Databases             209
                 Data Cleaning             210
      APIs and Data Collection             224
          Big Data and PySpark             242
           Supervised Learning             255
              Model Evaluation             240
           Feature Engineering             236
                Decision Trees             227
                 Random Forest             238
         Large Language Models             226
            Prompt Engineering             275
Retrieval Augmented Generation             274
                Pandas Library             252
            Data Visualization             249


In [9]:
documents = df['content'].tolist()
ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared: {len(documents)}")
print(f"Total Document Ids: {ids[0]}")
print(f"First Metadata: {metadatas[0]}")
print(f"First 100 chars of doc: {documents[0][:100]}...")

Total chunks prepared: 15
Total Document Ids: note_N001
First Metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc: ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [10]:
#Loading Embedding Model

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
test_embedding = embedding_model.encode("This is a test sentence.")
print(f"Shape of embedding: {test_embedding.shape}")
print(f"First 5 values of embedding: {test_embedding[:5]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embedding: (384,)
First 5 values of embedding: [0.08429647 0.05795366 0.00449333 0.1058211  0.00708344]


In [11]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="college_rag_notes")

print("Collection Created")
print(f"Collection name: {collection.name}")
print(f"Number of Documents in Collection: {collection.count()}")

Collection Created
Collection name: college_rag_notes
Number of Documents in Collection: 0


In [12]:
print("Generating embeddings for all notes")
embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\nEmbedding Matrix shape: {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas,
    embeddings=embeddings_list
)
print("Documents successfully added to chromaDB")
print(f"Total documents on collection: {collection.count()}")

Generating embeddings for all notes


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding Matrix shape: (15, 384)
Documents successfully added to chromaDB
Total documents on collection: 15


In [13]:
def retrieve_relevent_chunks(question, top_k=3):
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )
  return results
print("Retrieval function defined successfully")

Retrieval function defined successfully


In [14]:
test_question = input("Enter Your Question: ")
print(f"Test Question: {test_question}")
print("="*60)

results = retrieve_relevent_chunks(test_question, top_k=3)
print("\nTop 3 Retrieved Notes:")
print("="*60)

for i,(doc, dist, meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
)):
  print(f"Note {i+1}:")
  print(f"Subject  : {meta['subject']}")
  print(f"Topic    : {meta['topic']}")
  print(f"Distance : {dist}")
  print(f"Content  : {doc[:120]}...\n")

Enter Your Question: what is ETL
Test Question: what is ETL

Top 3 Retrieved Notes:
Note 1:
Subject  : Data Engineering
Topic    : ETL Pipelines
Distance : 0.3394540250301361
Content  : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Note 2:
Subject  : Data Engineering
Topic    : APIs and Data Collection
Distance : 1.5857576131820679
Content  : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Note 3:
Subject  : Generative AI
Topic    : Retrieval Augmented Generation
Distance : 1.6214964389801025
Content  : RAG or Retrieval Augmented Generation is a technique where an AI model first retrieves relevant documents from a knowled...



In [15]:
def build_context_from_results(results):
  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts

In [16]:
def generate_rag_answer(question,context):
  system_prompt = """You are helpful academic assistent for Engineering Students"""
  user_prompt = f"""
  Context from knowledge base:
  {context}
  ---
  Student's Question:{question}
  Please answer the question based only on the context provided above.
  """

  response = groq_client.chat.completions.create(
      model = "llama-3.1-8b-instant",
      messages = [
          {
          "role": "system",
          "content": system_prompt
          },
          {
          "role": "user",
          "content": user_prompt
          }],
      temperature = 0.1,
      max_tokens = 500
  )
  return response.choices[0].message.content

In [17]:
def ask_college_assistant(question, top_k=3, verbose=True):
    if verbose:
      print("+"*60)
      print(f"COLLEGE ASSISTANT")
      print("+"*60,'\n')
      print(f"You aksed, {question}\n")
      print("Step 1: Retrieving Relevant Chunks...")
      print("="*60)
      results = retrieve_relevent_chunks(question, top_k=top_k)
      print("Step 2: Building Context...")
      print("="*60)
      context = build_context_from_results(results)
      print("Step 3: Generating Answer...")
      print("="*60)
      answer = generate_rag_answer(question, context)

    return answer

In [18]:
print(ask_college_assistant(question=input("Enter your query: ")))

Enter your query: what is ETL
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
COLLEGE ASSISTANT
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++ 

You aksed, what is ETL

Step 1: Retrieving Relevant Chunks...
Step 2: Building Context...
Step 3: Generating Answer...
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.
